# 1、HumanInTheLoopMiddleware中间件

## 1.1 举例的过程1：工具调用的中断

In [1]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

# 从.env文件中加载环境变量
load_dotenv(override=True)

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")

model = init_chat_model(
    model="deepseek-v4-flash",
    model_provider="deepseek",
    #profile={"max_input_tokens":128_000},
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL,
    extra_body={"thinking": {"type": "disabled"}}
)

In [2]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain.messages import HumanMessage
from langchain.tools import tool
from langgraph.types import Command
from rich import print as rprint


@tool
def get_weather(city: str, is_forcast: bool = False) -> str:
    """
    查询指定城市天气

    Args:
        city: 城市名称
        is_forcast: 是否包含明日天气预报？
    """
    res = f"{city}今天天气不错"
    if is_forcast:
        res += "\n明天下雨"
    return res


@tool
def get_news() -> str:
    """
    查询当日新闻
    """
    return "中方三艘油轮通过霍尔木兹海峡"


@tool
def read_email_tool(email_id: str) -> str:
    """通过邮件ID读取内容的伪函数"""
    return f"邮件ID：{email_id}\n是空的"


@tool
def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """发送邮件伪函数"""
    print(">>> 真的执行发送邮件工具了")
    return f"发送给 {recipient} 的邮件标题是：{subject}，内容：{body}"


agent = create_agent(
    model=model,
    tools=[get_weather, get_news, read_email_tool, send_email_tool],
    checkpointer=InMemorySaver(),  # 通过该参数启用短期记忆，因为终端后还需要接着之前的位置执行，如果重启会话没有之前的执行进度与记忆
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "get_weather": True, # True表示会中断，然后三种选项全部开放给用户选
                "get_news": True,
                "read_email_tool": False,  # False表示不会中断，调用到该工具不会触发HumanInTheLoopMiddleWare中间件
                # 这种写法表示会中断，指定了开放给用户的选项有approve跟reject，并没有edit的选项
                "send_email_tool": {
                    "allowed_decisions": ["approve", "reject"],  # 中断策略
                    "description": "发送邮件中断了..."   # 中断的描述信息
                },
            },
            description_prefix="中断啦！！"
        ),
    ]
)

config = {"configurable": {"thread_id": "1"}}  ## 也是为了实现短期记忆，保证继续执行在同一个会话当中，通过thread_id指定

response = agent.invoke({
    "messages": [HumanMessage(content="请帮我查询今天北京的天气"
                                      "查询今日新闻"
                                      "查看ID为 'sk2131421' 的邮件内容，"
                                      "向15641685664@qq.com发送邮件，标题是'哈哈哈'，内容是：'你好啊'"
                                      "同时做这四件事")]
},
    config=config  # 初始化config
)


rprint(response)

{
    'messages': [
        HumanMessage(
            content="请帮我查询今天北京的天气查询今日新闻查看ID为 'sk2131421' 
的邮件内容，向15641685664@qq.com发送邮件，标题是'哈哈哈'，内容是：'你好啊'同时做这四件事",
            additional_kwargs={},
            response_metadata={},
            id='1cd57473-a5db-4fb8-bde2-1890a99e6e2d'
        ),
        AIMessage(
            content='我来同时为您完成这四件事：查询北京天气、查看今日新闻、读取指定邮件、发送邮件。',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 190,
                    'prompt_tokens': 517,
                    'total_tokens': 707,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0},
                    'prompt_cache_hit_tokens': 0,
                    'prompt_cache_miss_tokens': 517
                },
                'model_provider': 'deepseek',
                'model_name': 'deepseek-v4-flash',
                'system_fingerprint': 'a26a7955944dc5c60445bff77fac9c8e',
                'id': '4b9bb0ff-b8da-4f10-998c-e8e7ac10c6ad',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a0478f-98ca-7213-aace-25655e098bfb-0',
            tool_calls=[
                {
                    'name': 'get_weather',
                    'args': {'city': '北京'},
                    'id': 'call_00_FX5feOa9OmxhAeDin53X8608',
                    'type': 'tool_call'
                },
                {'name': 'get_news', 'args': {}, 'id': 'call_01_wAxqLWwqGOaEVP88QvoL9958', 'type': 'tool_call'},
                {
                    'name': 'read_email_tool',
                    'args': {'email_id': 'sk2131421'},
                    'id': 'call_02_UfhBqh8rmW2COni8jjDT1657',
                    'type': 'tool_call'
                },
                {
                    'name': 'send_email_tool',
                    'args': {'recipient': '15641685664@qq.com', 'subject': '哈哈哈', 'body': '你好啊'},
                    'id': 'call_03_kltIxY94sQ0tScpUksmB9133',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 517,
                'output_tokens': 190,
                'total_tokens': 707,
                'input_token_details': {'cache_read': 0},
                'output_token_details': {}
            }
        )
    ],
    '__interrupt__': [
        Interrupt(
            value={
                'action_requests': [
                    {
                        'name': 'get_weather',
                        'args': {'city': '北京'},
                        'description': "中断啦！！\n\nTool: get_weather\nArgs: {'city': '北京'}"
                    },
                    {'name': 'get_news', 'args': {}, 'description': '中断啦！！\n\nTool: get_news\nArgs: {}'},
                    {
                        'name': 'send_email_tool',
                        'args': {'recipient': '15641685664@qq.com', 'subject': '哈哈哈', 'body': '你好啊'},
                        'description': '发送邮件中断了...'
                    }
                ],
                'review_configs': [
                    {'action_name': 'get_weather', 'allowed_decisions': ['approve', 'edit', 'reject']},
                    {'action_name': 'get_news', 'allowed_decisions': ['approve', 'edit', 'reject']},
                    {'action_name': 'send_email_tool', 'allowed_decisions': ['approve', 'reject']}
                ]
            },
            id='c3b4cfe905dfa24a9f0394ef7e5a1c99'
        )
    ]
}

## 举例的过程2：指明工具调用请求决策


In [3]:

## 这里是在模拟人工对于中断后的操作
weather_decision = {
    "type" : "edit",
    "edited_action" : {
        "name" : "get_weather",
        "args" : {"city" : "上海市","is_forcast" : True},
    }
}


news_decision = {
    "type" : "approve"
}


send_email_decision = {
    "type" : "approve"
}


## 定义结果，存放每种中断人工处理的返回值
decisions = {
    "decisions" : []
}


## 获取中断的工具信息
interrupts = response.get("__interrupt__",[])
action_requests = interrupts[0].value["action_requests"]

## 循环工具中的中断信息，将前面模拟的人工处理的方式每种数据加入到相应的
for action_request in action_requests:
    if action_request["name"] == "get_weather":
        decisions["decisions"].append(weather_decision)
    if action_request["name"] == "get_news":
        decisions["decisions"].append(news_decision)
    if action_request["name"] == "send_email_tool":
        decisions["decisions"].append(send_email_decision)

if interrupts :
    resumed_response = agent.invoke(
        Command(resume=decisions),
        config = config
    )

    for msg in resumed_response["messages"]:
        msg.pretty_print()

>>> 真的执行发送邮件工具了
================================ Human Message =================================

请帮我查询今天北京的天气查询今日新闻查看ID为 'sk2131421' 的邮件内容，向15641685664@qq.com发送邮件，标题是'哈哈哈'，内容是：'你好啊'同时做这四件事
================================== Ai Message ==================================

我来同时为您完成这四件事：查询北京天气、查看今日新闻、读取指定邮件、发送邮件。
Tool Calls:
  get_weather (call_00_FX5feOa9OmxhAeDin53X8608)
 Call ID: call_00_FX5feOa9OmxhAeDin53X8608
  Args:
    city: 上海市
    is_forcast: True
  get_news (call_01_wAxqLWwqGOaEVP88QvoL9958)
 Call ID: call_01_wAxqLWwqGOaEVP88QvoL9958
  Args:
  read_email_tool (call_02_UfhBqh8rmW2COni8jjDT1657)
 Call ID: call_02_UfhBqh8rmW2COni8jjDT1657
  Args:
    email_id: sk2131421
  send_email_tool (call_03_kltIxY94sQ0tScpUksmB9133)
 Call ID: call_03_kltIxY94sQ0tScpUksmB9133
  Args:
    recipient: 15641685664@qq.com
    subject: 哈哈哈
    body: 你好啊
================================= Tool Message =================================
Name: get_weather

上海市今天天气不错
明天下雨
================================